# 01 - Data audit

What is in the raw Lending Club file, before anything is changed: row counts, column types, missing values, the `loan_status` breakdown, and the checks behind the modelling decisions (which loans, which years, which columns).

Reads `data/interim/loans_raw.parquet` (written by `python run_pipeline.py --from ingest`). Profiling code lives in `credit_risk.data.audit`. The analysis queries are SQL files in `sql/analysis/`, run here with `audit.run_sql_file`. Each section says which decision its result supports; those decisions are recorded in `docs/cleaning_log.md`.

In [1]:
import pandas as pd

from credit_risk.data import audit

pd.set_option("display.max_rows", None)
con = audit.open_raw_loans()

## Row counts

`non_loan_rows` are footer lines in the CSV (non-numeric `id`). They are excluded from every table below.

In [2]:
audit.row_counts(con)

{'total_rows': 2260701, 'loan_rows': 2260668, 'non_loan_rows': 33}

## Column types and missing values

The raw table is all text, so `inferred_type` names the one format every non-missing value follows (`numeric`, `percent_text`, `mon_year_text`, `text`, `mixed` or `empty`). Sorted by % missing.

In [3]:
profile = audit.profile_columns(con)
profile.sort_values("pct_missing", ascending=False).reset_index(drop=True)

,column,inferred_type,n_missing,pct_missing,n_distinct
0,member_id,empty,2260668,100.00,0
1,orig_projected_additional_accrued_interest,numeric,2252017,99.62,7487
2,hardship_reason,text,2249751,99.52,9
3,hardship_payoff_balance_amount,numeric,2249751,99.52,10893
4,hardship_last_payment_amount,numeric,2249751,99.52,9045
5,payment_plan_start_date,mon_year_text,2249751,99.52,27
6,hardship_type,text,2249751,99.52,1
7,hardship_status,text,2249751,99.52,3
8,hardship_start_date,mon_year_text,2249751,99.52,27
9,deferral_term,numeric,2249751,99.52,1


In [4]:
profile["inferred_type"].value_counts()

inferred_type
numeric          113
text              24
mon_year_text     11
mixed              2
empty              1
Name: count, dtype: int64

In [5]:
print(f"{(profile['pct_missing'] > 50).sum()} of {len(profile)} columns are more than 50% missing")

44 of 151 columns are more than 50% missing


## loan_status

Value counts of the target source, with the outcome each status maps to (`default`, `fully_paid`, or `unresolved`, which is dropped before modelling). Query: `02_status_breakdown.sql`.

In [6]:
status = audit.run_sql_file(con, "02_status_breakdown")
status

,status,n_loans,pct,outcome
0,Fully Paid,1076751,47.63,fully_paid
1,Current,878317,38.85,unresolved
2,Charged Off,268559,11.88,default
3,Late (31-120 days),21467,0.95,unresolved
4,In Grace Period,8436,0.37,unresolved
5,Late (16-30 days),4349,0.19,unresolved
6,Does not meet the credit policy. Status:Fully ...,1988,0.09,fully_paid
7,Does not meet the credit policy. Status:Charge...,761,0.03,default
8,Default,40,0.00,default


In [7]:
status.groupby("outcome")[["n_loans", "pct"]].sum()

,n_loans,pct
outcome,,
default,269360,11.91
fully_paid,1078739,47.72
unresolved,912569,40.36


## Loans and default rate by issue year

Query: `01_loans_per_year.sql`. Default rate is among resolved loans only. Read it together with the section on unfinished loans below: part of the rise from 15.6% for 2013 to 23% for 2016 is an artefact of loans that have not finished yet.

In [8]:
audit.run_sql_file(con, "01_loans_per_year")

,issue_year,n_loans,n_default,n_resolved,default_rate
0,2007,603,158,603,0.2620
1,2008,2393,496,2393,0.2073
2,2009,5281,723,5281,0.1369
3,2010,12537,1757,12537,0.1401
4,2011,21721,3297,21721,0.1518
5,2012,53367,8644,53367,0.1620
6,2013,134814,21024,134804,0.1560
7,2014,235629,41162,223103,0.1845
8,2015,421095,75804,375546,0.2019
9,2016,434407,68252,293105,0.2329


## Default rate by grade

Query: `03_default_rate_by_grade.sql`, all resolved loans.

**Finding:** the default rate rises steadily from 6.0% for grade A to 49.7% for grade G, so Lending Club's grade is a strong risk ranking. **Decision:** hold `grade`, `sub_grade` and `int_rate` out of the headline model, because using them would partly copy Lending Club's own model. They are added back only in the ablation run.

In [9]:
audit.run_sql_file(con, "03_default_rate_by_grade")

,grade,n_loans,default_rate
0,A,235193,0.0604
1,B,393102,0.1340
2,C,382323,0.2244
3,D,201657,0.3038
4,E,94192,0.3843
5,F,32306,0.4515
6,G,9326,0.4967


## Are recent loans fully resolved?

Query: `04_resolved_share_by_term_and_year.sql`. `pct_resolved` is the share of loans with a final outcome; `default_rate` is among those only.

**Finding:** 36-month loans are fully resolved up to 2015 (99.9%), but only 72% for 2016, 40% for 2017 and 12% for 2018. Their default rate jumps from 14.9% for 2015 to 19.8% for 2016 because the loans that have finished early skew towards early defaults. 60-month loans are less resolved still: 83% for 2014, 67% for 2015, 55% for 2016.

**Decision:** model 36-month loans only, and end the test set at December 2015.

In [10]:
audit.run_sql_file(con, "04_resolved_share_by_term_and_year")

,issue_year,term_months,n_loans,n_resolved,pct_resolved,default_rate
0,2007,36,603,603,100.0,0.2620
1,2008,36,2393,2393,100.0,0.2073
2,2009,36,5281,5281,100.0,0.1369
3,2010,36,9156,9156,100.0,0.1092
4,2010,60,3381,3381,100.0,0.2239
5,2011,36,14101,14101,100.0,0.1063
6,2011,60,7620,7620,100.0,0.2360
7,2012,36,43470,43470,100.0,0.1358
8,2012,60,9897,9897,100.0,0.2770
9,2013,36,100422,100422,100.0,0.1233


## Which columns exist in which years?

Query: `05_missingness_by_year.sql`. Percent of loans with the column empty. `last_delinq_control` is a control column.

**Finding:** `tot_cur_bal` and `mort_acc` (the bureau-snapshot columns) are empty for every loan issued 2007 to 2011 and for about half of 2012. Three months-since columns (`last_major_derog`, `recent_bc_dlq`, `recent_revol_delinq`) do not exist before 2012. `open_acc_6m`, `il_util` and `inq_last_12m` are about 95% empty through 2015 and filled in only from 2016. The control column is partly empty in every year, because there empty means the event never happened.

**Decisions:** start modelling at 2013; drop the columns first collected in late 2015; do not add missing-value flags to the structurally missing columns, because a flag would only encode the issue year.

In [11]:
audit.run_sql_file(con, "05_missingness_by_year")

,issue_year,n_loans,tot_cur_bal,mort_acc,last_major_derog,recent_bc_dlq,recent_revol_delinq,open_acc_6m,il_util,inq_last_12m,last_delinq_control
0,2007,603,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,6.5
1,2008,2393,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,38.1
2,2009,5281,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,67.0
3,2010,12537,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,64.3
4,2011,21721,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,66.2
5,2012,53367,52.0,14.0,89.9,87.4,73.9,100.0,100.0,100.0,58.8
6,2013,134814,0.0,0.0,79.9,77.7,69.9,100.0,100.0,100.0,56.5
7,2014,235629,0.0,0.0,71.8,73.6,64.0,100.0,100.0,100.0,49.2
8,2015,421095,0.0,0.0,70.9,74.2,64.0,94.9,95.6,94.9,48.4
9,2016,434407,0.0,0.0,71.1,74.4,63.7,0.0,13.2,0.0,47.2


## Joint applications

Query: `06_application_type_by_year.sql`.

**Finding:** there are no joint applications before 2015, then 0.1% of loans in 2015, 2.0% in 2016, 9.6% in 2017 and 13.9% in 2018. A model trained on 2013 to 2014 has never seen one, and on them `annual_inc` and `dti` describe one borrower while the joint columns describe the pair.

**Decision:** model Individual loans only.

In [12]:
audit.run_sql_file(con, "06_application_type_by_year")

,issue_year,n_loans,n_joint,pct_joint
0,2007,603,0,0.0
1,2008,2393,0,0.0
2,2009,5281,0,0.0
3,2010,12537,0,0.0
4,2011,21721,0,0.0
5,2012,53367,0,0.0
6,2013,134814,0,0.0
7,2014,235629,0,0.0
8,2015,421095,511,0.1
9,2016,434407,8789,2.0


## Modelling population and split

Query: `07_modelling_population.sql`, with the filters and dates read from `config.py`.

**Finding:** the three splits have close default rates (13.2%, 15.1% and 14.7%), so they can be compared. The 386,155 loans issued after 2015 have a 19.2% default rate that is partly the artefact above, so they are excluded.

These counts are the ones recorded in rows 4 to 10 of `docs/cleaning_log.md`. Week 3's `validate.py` should assert the same on the cleaned table.

In [13]:
funnel = audit.run_sql_file(con, "07_modelling_population")
funnel

,step_no,step,n_loans,default_rate
0,1,All loan rows,2260668,NaN
1,2,"Resolved (Fully Paid, Charged Off, Default)",1348099,0.1998
2,3,... and the modelling term,1023206,0.1602
3,4,... and the modelling application type,1006938,0.1594
4,5,... and issued from the training start,931934,0.1618
5,6,Train,262992,0.1319
6,7,Validation,120790,0.1512
7,8,Test,161997,0.1470
8,9,Issued after the test period (excluded),386155,0.1915


## Summary

The audit is done when you can state the row count, date range, default rate and the top data problems.

In [14]:
counts = audit.row_counts(con)
first, last = con.execute("SELECT min(issue_date), max(issue_date) FROM loans_audit").fetchone()
resolved_default_rate = funnel.set_index("step_no").loc[2, "default_rate"]
print(f"{counts['loan_rows']:,} loans issued {first:%b %Y} to {last:%b %Y}")
print(f"Default rate among resolved loans: {resolved_default_rate:.1%}")

2,260,668 loans issued Jun 2007 to Dec 2018
Default rate among resolved loans: 20.0%


**Top data problems**

1. **Unfinished loans bias recent years.** Only 36-month loans issued up to 2015 are fully resolved, so the model is built and tested on those.
2. **Columns that exist only in some years.** The bureau-snapshot columns start in 2013 and another group in late 2015. 44 of the 151 columns are more than 50% missing.
3. **Leakage.** 39 columns are only known after the loan is issued; they are never features, and seven are kept in a separate outcome table for the profit curve.
4. **Joint applications start in 2015**, so a model trained on earlier years has never seen one.
5. **A raw table that is all text**, with text dates such as `Dec-2015`, text `emp_length` such as `10+ years`, and 33 non-loan footer lines.

Every column's decision and reason is in `docs/column_classification.csv`.